In [1]:
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from torch.optim.adamw import AdamW
from sklearn.metrics import accuracy_score, f1_score
import os
import pandas as pd

2025-05-14 08:12:06.878926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747181526.970398    2852 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747181526.996635    2852 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747181527.369566    2852 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747181527.369588    2852 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747181527.369592    2852 computation_placer.cc:177] computation placer alr

In [2]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/kurty/Project/lib/python3.12/site-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [3]:
class MELDDataset(Dataset):
    def __init__(self, data, target_sr=16000):
        """
        data: list of (audio_path, label)
        """
        self.data = data
        self.target_sr = target_sr

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio_path, label = self.data[idx]
        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.squeeze(0)

        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sr)
            waveform = resampler(waveform)

        if waveform.shape[0] < self.target_sr:
            return self.__getitem__((idx + 1) % len(self.data))

        return {
            "input_values": waveform,
            "label": label
        }


In [4]:
def collate_fn_wav2vec2(batch):
    waveforms = [item[0].tolist() for item in batch]  # convert to list of floats
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)

    padded = processor.pad(
        {"input_values": waveforms},
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,  # ✅ force creation
    )
    
    return padded.input_values, padded.attention_mask, labels

In [5]:
def get_list(csv, path, maps):
    data = pd.read_csv(csv)
    item_list = []
    
    for _, row in data.iterrows():
        utt_id = row['Utterance_ID']
        dia_id = row['Dialogue_ID']
        sentiment = row['Sentiment']
        label = maps[sentiment]
        
        audio_path = os.path.join(path, f"dia{dia_id}_utt{utt_id}.wav")
        if os.path.exists(audio_path):
            item_list.append((audio_path, label))
        else:
            print(f'where is {audio_path}')
    return item_list

In [6]:
sentiment_map = {
    'neutral':0,
    'positive':1,
    'negative':2
}

In [7]:
train_list = get_list('MELD.Raw/train_sent_emo.csv', 'MELD.Wav/train', sentiment_map)
test_list = get_list('MELD.Raw/test_sent_emo.csv', 'MELD.Wav/test', sentiment_map)
val_list = get_list('MELD.Raw/dev_sent_emo.csv', 'MELD.Wav/dev', sentiment_map)

where is MELD.Wav/dev/dia110_utt7.wav


In [8]:
train_dataset = MELDDataset(train_list)
val_dataset = MELDDataset(val_list)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn_wav2vec2)
val_loader = DataLoader(val_dataset, batch_size=2, collate_fn=collate_fn_wav2vec2)

In [9]:
num_labels = 3
model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-base", num_labels=num_labels)
model.to(device)
model.config.mask_time_length = 2

/home/kurty/Project/lib/python3.12/site-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [11]:
def data_collator(features):
    input_values = [f["input_values"] for f in features]
    labels = [f["label"] for f in features]

    batch = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt",
        return_attention_mask=True
    )
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch

In [ ]:
from transformers import Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer

model = Wav2Vec2ForSequenceClassification.from_pretrained("facebook/wav2vec2-base", num_labels=7)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    learning_rate=1e-5,
    fp16=True,  # Optional: if your GPU supports it

)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2852/1742145870.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
